In [1]:
import numpy as np
import matplotlib.pyplot as plt

from lammps import lammps, LMP_STYLE_ATOM, LMP_TYPE_VECTOR, LMP_TYPE_ARRAY

#from lammps import lammps, LMP_STYLE_GLOBAL,LMP_STYLE_ATOM, LMP_STYLE_LOCAL, LMP_TYPE_SCALAR
#from lammps import LMP_TYPE_VECTOR, LMP_TYPE_ARRAY, LMP_SIZE_VECTOR, LMP_SIZE_ROWS, LMP_SIZE_COLS
from ctypes import c_double, c_int

In [2]:
#lmp = lammps()
lmp = lammps(cmdargs=['-screen', 'none'])

lmp.commands_string('''
# ------------------------ INITIALIZATION ----------------------------
processors    * * *
units         metal
dimension    3
boundary    p    p    p
atom_style   atomic
atom_modify map yes


box        tilt large
#--------------------------- LAMMPS Data File--------------------------
#read_data    input_data/thicker_domain.lmp
read_data    input_data/4b_step.lmp
change_box    all triclinic
reset_atom_ids sort yes

#--------------
pair_style    eam/alloy
pair_coeff    * * input_data/Cu_mishin1.eam.alloy Cu

#-------------------Various continuation commands----------------------
compute forces all property/atom fx fy fz
compute ids all property/atom id
compute x_check all property/atom x y z
#atom_modify map yes
''')

In [3]:
natoms = lmp.extract_global("natoms")
original_box_size = lmp.extract_box()
ref_X = np.reshape(
            np.array(lmp.gather_atoms("x", 1, 3)),
            (natoms, 3),
        ).copy()

# saving energies, configurations, predicted next step configurations, IDs of atoms, values of sigma and max forces:
#energies
E = []
#optimal configurations
X = []
# unnormalised approximate tangents:
X_tangents = []
# initial guess for the next step:
X_predictors = []

# IDs of atoms as assigned by LAMMPS
IDS = []

# values of continuation parameter
μs = []

#max force, unnused at the moment
max_force = []

#all forces
forces = []

# change in the parameter
increment = 0.1

In [4]:
original_box_size

([-64.3409885018, 0.0, 380.0],
 [107.36002035, 4.427452710080594, 500.0],
 0.0,
 0.0,
 0.0,
 [1, 1, 1],
 0)

In [6]:
ref_X[0]

array([-63.60395942,   2.60939334, 402.74950777])

In [7]:
# a quasi static run: adjust mu and minimise
for k in range(100):
    μ = 0.0 + (k)*increment
    print(μ)
    μs += [μ]    
    lammps_commands = f"""
     change_box all x final {original_box_size[0][0]+μ} {original_box_size[1][0]-μ} units box
    """
    lmp.commands_string(lammps_commands)
        
    print(lmp.extract_box()[:2])
    
    if k < 2:
        ref_X_flat = ref_X.flatten()
        ref_X_flat = ((3*natoms)*c_double)(*ref_X_flat)
        lmp.scatter_atoms("x", 1, 3, ref_X_flat)
        
        #lmp.command('write_dump all custom run_dump1 id type x y z xu yu zu')        
        
        #quick check:
        #print(ref_X[0])
        #check = np.reshape(
        #      np.array(lmp.gather_atoms("x", 1, 3)),
        #      (natoms, 3),
        # )
        #print(check[0])
        #lmp.command('write_dump all custom run_dump2 id type x y z xu yu zu')
        
    #minimise
    lmp.command('run 0')
    lmp.command('min_style cg')
    lmp.command('minimize 0 1e-5 3000 3000')
    _E = lmp.get_thermo("pe")
    _max_force = lmp.get_thermo("fmax")
    #_IDS = np.array(lmp.gather_atoms("id", 0, 1))
    _IDS = lmp.numpy.extract_compute('ids',LMP_STYLE_ATOM,LMP_TYPE_VECTOR).astype('int32')
    _F = lmp.numpy.extract_compute('forces',LMP_STYLE_ATOM,LMP_TYPE_ARRAY).copy()
    _X = np.reshape(
             np.array(lmp.gather_atoms("x", 1, 3)),
             (natoms, 3),
         )
    E += [_E]
    X += [_X.copy()]
    IDS += [_IDS.copy()]
    max_force += [_max_force]
    forces += [_F]
    print(max_force[-1])
    lmp.command('reset_timestep 0')
    lmp.command(f'write_dump all custom dumps/thin_domain/minimise_dump/minimise_dump_{len(X)} id type x y z xu yu zu')        

    #lmp.command('unfix relax')
    
# copy the results:
X_c = X.copy()
E_c = E.copy()
IDS_c = IDS.copy()
μs_c = μs.copy()
forces_c = forces.copy()
max_force_c = max_force.copy()
X_predictors_c = X_predictors.copy()

0.0
([-64.3409885018, 0.0, 380.0], [107.36002035, 4.427452710080594, 500.0])
6.212929019068503e-07
0.1
([-64.2409885018, 0.0, 380.0], [107.26002035, 4.427452710080594, 500.0])
3.7187752423284426e-07
0.2
([-64.1409885018, 0.0, 380.0], [107.16002035, 4.427452710080594, 500.0])
3.3185372173838434e-07
0.30000000000000004
([-64.0409885018, 0.0, 380.0], [107.06002035, 4.427452710080594, 500.0])
3.482034653563837e-07
0.4
([-63.9409885018, 0.0, 380.0], [106.96002035, 4.427452710080594, 500.0])
3.1583868634332823e-07
0.5
([-63.8409885018, 0.0, 380.0], [106.86002035, 4.427452710080594, 500.0])
3.60514723324757e-07
0.6000000000000001
([-63.7409885018, 0.0, 380.0], [106.76002035, 4.427452710080594, 500.0])
3.9707324837878755e-07
0.7000000000000001
([-63.640988501799995, 0.0, 380.0], [106.66002035, 4.427452710080594, 500.0])
3.2983225957796194e-07
0.8
([-63.5409885018, 0.0, 380.0], [106.56002035, 4.427452710080594, 500.0])
3.228979614951742e-07
0.9
([-63.4409885018, 0.0, 380.0], [106.46002035, 4.42